# TCP Data

---

### package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl
# import globus_sdk


In [2]:
from spectranorm import snm

In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


## Extracting data

---

In [4]:
data_info_df = pd.read_csv("/home/sina/storage/Normative_Modeling/data/csv/TCP_demogr_with_recon_all.csv")
data_info_df.shape


(241, 8)

In [ ]:
data_info_df.head(10)

In [6]:
data_info_df[['scanner']].value_counts(dropna=False)

scanner       
Siemens Prisma    241
Name: count, dtype: int64

In [8]:
data_info_df[['site', 'diagnosis']].value_counts(dropna=False)

site        diagnosis         
TCP_site_1  Patient               79
TCP_site_2  Patient               70
TCP_site_1  General Population    58
TCP_site_2  General Population    34
Name: count, dtype: int64

In [26]:
data_info_df[['site', 'scanner', 'diagnosis']].value_counts(dropna=False)

site        scanner         diagnosis         
TCP_site_1  Siemens Prisma  Patient               79
TCP_site_2  Siemens Prisma  Patient               70
TCP_site_1  Siemens Prisma  General Population    58
TCP_site_2  Siemens Prisma  General Population    34
Name: count, dtype: int64

In [ ]:
list(data_info_df["recon_all_path"][:10])

In [ ]:
from contextlib import suppress

sex_encoder = {
    'female': 'F',
    'male': 'M'
}

tcp_valid_subjects_dict = {}

for _, row in tqdm(data_info_df.iterrows()):
    if pd.notna(row["recon_all_path"]):
        key = row["subject_id"]
        with suppress(KeyError): # ignore dictionary misses
            tcp_valid_subjects_dict[key] = {
                "unique_id": key,
                "participant_id": key,
                "session_id": "V1_fs",
                "sex": sex_encoder[row["sex"]],
                "age": row["age_in_years"],
                "site": row["site"],
                "scan_path": row["recon_all_path"],
                "diagnosis": (row["diagnosis"] == "Patient"),
            }

len(tcp_valid_subjects_dict), list(tcp_valid_subjects_dict.items())[:1]


In [12]:
items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]

def directory_is_valid(path):
    return len([f for f in items if (path / f).exists()]) == len(items)

In [13]:
# Store high-resolution thickness for each individual in a separate file
for idx, subject in enumerate(tqdm(tcp_valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]

    freesurfer_directory = tcp_valid_subjects_dict[subject]["scan_path"]
    
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/TCP/{sub_dir}/{subject}.thickness.fslr.npy"

    if (directory_is_valid(Path(freesurfer_directory) / "surf")) and (not Path(thickness_fslr_output).exists()):
        # Compute fslr thickness
        transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(freesurfer_directory)
        np.save(
            ensure_dir(thickness_fslr_output),
            transformed_fslr_thickness.astype(np.float32)
        )


  0%|          | 0/237 [00:00<?, ?it/s]

In [ ]:
%%time
for idx, subject in enumerate(tqdm(tcp_valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]
    tcp_valid_subjects_dict[subject]["subject_index"] = idx
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/TCP/{sub_dir}/{subject}.thickness.fslr.npy"
    if Path(thickness_fslr_output).exists():
        tcp_valid_subjects_dict[subject]["thickness"] = np.load(
            thickness_fslr_output,
        ).mean()
    else:
        tcp_valid_subjects_dict[subject]["thickness"] = np.nan

len(tcp_valid_subjects_dict), list(tcp_valid_subjects_dict.items())[:1]


In [29]:
eno_items = [
    "lh.orig.nofix", "rh.orig.nofix",
]

# Compute Euler Number
for idx, subject in enumerate(tqdm(tcp_valid_subjects_dict)):
    if "euler_no" not in tcp_valid_subjects_dict[subject]:
        sub_dir = f"{idx:02d}"[-2:]
        freesurfer_directory = tcp_valid_subjects_dict[subject]["scan_path"]

        # Compute euler number
        tcp_valid_subjects_dict[subject]["euler_no"] = snm.utils.nitools.compute_total_euler_number(
            Path(freesurfer_directory)
        )


  0%|          | 0/237 [00:00<?, ?it/s]

In [30]:
# Validity checks
for idx, subject in enumerate(tqdm(tcp_valid_subjects_dict)):
    if "validity_check" not in tcp_valid_subjects_dict[subject]:
        tcp_valid_subjects_dict[subject]["validity_check"] = (
            (tcp_valid_subjects_dict[subject]["diagnosis"] == False)  # Exclude those with a diagnosis
            and
            (tcp_valid_subjects_dict[subject]["thickness"] != np.nan)  # Exclude those missing thickness data
            and
            (tcp_valid_subjects_dict[subject]["euler_no"] != np.nan)  # Exclude those missing euler number
        )


  0%|          | 0/237 [00:00<?, ?it/s]

In [ ]:
len(tcp_valid_subjects_dict), list(tcp_valid_subjects_dict.items())[:1]


In [31]:
import joblib

joblib.dump(tcp_valid_subjects_dict, ensure_dir("/home/sina/storage/Normative_Modeling/data/datasets/TCP/subjects.joblib"))


['/home/sina/storage/Normative_Modeling/data/datasets/TCP/subjects.joblib']

In [ ]:
import joblib

dataset = "TCP"

# Load the dictionary
valid_subjects_dict = joblib.load(
    f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"
)

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [ ]:
final_df = pd.DataFrame({
    'age': [valid_subjects_dict[key]["age"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'thickness': [valid_subjects_dict[key]["thickness"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'sex': [valid_subjects_dict[key]["sex"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'site': [valid_subjects_dict[key]["site"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_ID': [valid_subjects_dict[key]["participant_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'euler_no': [valid_subjects_dict[key]["euler_no"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_folder': [valid_subjects_dict[key]["unique_id"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
    'subject_index': [valid_subjects_dict[key]["subject_index"] for key in valid_subjects_dict if valid_subjects_dict[key]["validity_check"]],
})
final_df['dataset'] = dataset
final_df.head(), final_df.shape


In [7]:
# randomly select only one timepoint per subject (cross-sectional sample)
final_df_subset = final_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# # Only one site:
# final_df_subset.to_parquet(
#     ensure_dir(f'/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/demography.parquet')
# )

# final_df_subset.shape

# Multiple sites:
# Keep only sites with at least 15 subjects
subjects_per_site = final_df_subset.groupby("site")["subject_ID"].nunique()
valid_sites = subjects_per_site[subjects_per_site >= 15].index

final_df_subset[final_df_subset["site"].isin(valid_sites)].to_parquet(
    ensure_dir(f'/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/demography.parquet')
)

final_df_subset[final_df_subset["site"].isin(valid_sites)].shape


(91, 9)

# ✅ Finished!
